In [1]:
# ============================================================
# MD7 — DISTILROBERTA + TARGETED TELLING-CLASS WEIGHT
#
# Research question:
#   MD6 (alpha=0.75) became the best single model, mainly by
#   improving focus F1, but telling F1 fell relative to MD4.
#   Can we recover some telling performance without losing the
#   MD6 focus/accuracy gain?
#
# BASE CONFIGURATION:
#   Start from MD6 alpha=0.75 softened inverse-frequency weights.
#
# ONLY EXPERIMENTAL CHANGE FROM MD6:
#   Increase ONLY the telling-class weight halfway from the
#   MD6 value toward the original MD4 full-weight value.
#
# Everything else is held fixed:
#   Model: distilbert/distilroberta-base
#   Max length: 512
#   LR: 2e-5
#   Effective global batch: 16 (2 GPUs x batch 8)
#   Epochs: 3
#   Weight decay: 0.01
#   Warmup steps: 280
#   Seed: 42
#
# Primary metric: Macro-F1
# Secondary metric: Accuracy
# Test set: NOT USED
# ============================================================

import gc
import json
import random
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F

from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

SEED = 42
MODEL_NAME = "distilbert/distilroberta-base"
MAX_LENGTH = 512

BASE_WEIGHT_ALPHA = 0.75
TELLING_INTERPOLATION = 0.50  # halfway from MD6 alpha=.75 to MD4 alpha=1.0

TRAIN_PATH = Path(
    "/kaggle/input/datasets/malindus/mathdial-move-selection/train.jsonl"
)
VAL_PATH = Path(
    "/kaggle/input/datasets/malindus/mathdial-move-selection/validation.jsonl"
)

OUTPUT_DIR = Path("/kaggle/working/md7_targeted_telling_weight")
RESULT_DIR = Path("/kaggle/working/md7_results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ["generic", "probing", "focus", "telling"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 78)
print("MD7 CONFIGURATION")
print("=" * 78)
print("Model:", MODEL_NAME)
print("Max length:", MAX_LENGTH)
print("Base weight alpha:", BASE_WEIGHT_ALPHA)
print("Telling interpolation:", TELLING_INTERPOLATION)
print("GPU count:", torch.cuda.device_count())

assert torch.cuda.is_available(), "GPU is required."
assert torch.cuda.device_count() == 2, (
    "This experiment is defined for Kaggle 2xT4. "
    f"Detected {torch.cuda.device_count()} GPU(s)."
)

def load_jsonl(path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(TRAIN_PATH)
val_rows = load_jsonl(VAL_PATH)

assert len(train_rows) == 14905
assert len(val_rows) == 1850

required = {"example_id", "qid", "problem", "history", "target_move"}
for name, rows in [("train", train_rows), ("validation", val_rows)]:
    assert required.issubset(rows[0].keys()), f"{name} schema mismatch"
    bad = [r for r in rows if r["target_move"] not in LABEL2ID]
    assert not bad, f"{name}: invalid labels found"

print("Train examples:", len(train_rows))
print("Validation examples:", len(val_rows))
print("Test set used: False")


MD7 CONFIGURATION
Model: distilbert/distilroberta-base
Max length: 512
Base weight alpha: 0.75
Telling interpolation: 0.5
GPU count: 2
Train examples: 14905
Validation examples: 1850
Test set used: False


In [2]:
# ============================================================
# INPUT FORMAT + TOKENIZATION + SOFTENED CLASS WEIGHTS
# ============================================================

def format_history(history):
    if not history:
        return "[No previous conversation]"
    return "\n".join(
        f"{turn['user']}: {turn['text']}"
        for turn in history
    )

def build_problem_text(example):
    return "Problem:\n" + str(example["problem"])

def build_conversation_text(example):
    return (
        "Conversation:\n"
        + format_history(example["history"])
        + "\n\nNext teacher pedagogical move:"
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.truncation_side = "left"

def prepare_rows(rows):
    prepared = []
    for ex in rows:
        prepared.append({
            "example_id": ex["example_id"],
            "qid": ex["qid"],
            "problem_text": build_problem_text(ex),
            "conversation_text": build_conversation_text(ex),
            "labels": LABEL2ID[ex["target_move"]],
        })
    return prepared

train_prepared = prepare_rows(train_rows)
val_prepared = prepare_rows(val_rows)

# Fixed DistilRoBERTa short/long diagnostic split.
val_original_lengths = []
for ex in val_rows:
    encoded = tokenizer(
        build_problem_text(ex),
        build_conversation_text(ex),
        add_special_tokens=True,
        truncation=False,
    )
    val_original_lengths.append(len(encoded["input_ids"]))

val_original_lengths = np.asarray(val_original_lengths)
short_mask = val_original_lengths <= 512
long_mask = val_original_lengths > 512

print("Validation <=512:", int(short_mask.sum()))
print("Validation >512:", int(long_mask.sum()))
assert int(short_mask.sum()) == 1507
assert int(long_mask.sum()) == 343

# ------------------------------------------------------------
# MD7 targeted class-weight experiment
#
# 1) Compute the same raw inverse-frequency weights as MD4.
# 2) Soften all classes using alpha=0.75 (the MD6 setting).
# 3) Change ONLY telling:
#       telling_MD7 = telling_MD6
#                     + 0.50 * (telling_MD4 - telling_MD6)
#
# This isolates the effect of a modest increase in telling weight.
# ------------------------------------------------------------

train_counts = Counter(r["target_move"] for r in train_rows)
n = len(train_rows)
k = len(LABELS)

raw_class_weights = np.asarray(
    [n / (k * train_counts[label]) for label in LABELS],
    dtype=np.float32,
)

md6_class_weights = np.power(
    raw_class_weights, BASE_WEIGHT_ALPHA
).astype(np.float32)

class_weights = md6_class_weights.copy()

telling_idx = LABEL2ID["telling"]
class_weights[telling_idx] = (
    md6_class_weights[telling_idx]
    + TELLING_INTERPOLATION
    * (raw_class_weights[telling_idx] - md6_class_weights[telling_idx])
)

print("\n" + "=" * 78)
print("CLASS WEIGHTS")
print("=" * 78)
for label, raw_w, md6_w, md7_w in zip(
    LABELS, raw_class_weights, md6_class_weights, class_weights
):
    print(
        f"{label:10s} count={train_counts[label]:4d} "
        f"raw_MD4={raw_w:.6f} MD6={md6_w:.6f} MD7={md7_w:.6f}"
    )

assert np.allclose(
    class_weights[:3], md6_class_weights[:3]
), "Only telling weight is allowed to change in MD7."

print(
    f"\nTelling weight: MD6={md6_class_weights[telling_idx]:.6f} "
    f"-> MD7={class_weights[telling_idx]:.6f} "
    f"(MD4 raw={raw_class_weights[telling_idx]:.6f})"
)

train_ds = Dataset.from_list(train_prepared)
val_ds = Dataset.from_list(val_prepared)

def tokenize_batch(batch):
    return tokenizer(
        batch["problem_text"],
        batch["conversation_text"],
        truncation="only_second",
        max_length=MAX_LENGTH,
    )

train_tok = train_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["problem_text", "conversation_text", "example_id", "qid"],
)

val_tok = val_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=["problem_text", "conversation_text", "example_id", "qid"],
)

assert max(len(x) for x in train_tok["input_ids"]) <= MAX_LENGTH
assert max(len(x) for x in val_tok["input_ids"]) <= MAX_LENGTH

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("\nTokenization: PASSED")


config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (578 > 512). Running this sequence through the model will result in indexing errors


Validation <=512: 1507
Validation >512: 343

CLASS WEIGHTS
generic    count=3579 raw_MD4=1.041143 MD6=1.030701 MD7=1.030701
probing    count=3413 raw_MD4=1.091781 MD6=1.068075 MD7=1.068075
focus      count=5443 raw_MD4=0.684595 MD6=0.752619 MD7=0.752619
telling    count=2470 raw_MD4=1.508603 MD6=1.361229 MD7=1.434916

Telling weight: MD6=1.361229 -> MD7=1.434916 (MD4 raw=1.508603)


Map:   0%|          | 0/14905 [00:00<?, ? examples/s]

Map:   0%|          | 0/1850 [00:00<?, ? examples/s]


Tokenization: PASSED


In [3]:
# ============================================================
# MODEL + WEIGHTED TRAINER + METRICS
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    label2id=LABEL2ID,
    id2label=ID2LABEL,
)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights_tensor = torch.tensor(
            class_weights, dtype=torch.float32
        )

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
    ):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights_tensor.to(logits.device)

        loss = F.cross_entropy(
            logits,
            labels,
            weight=weights,
        )

        if return_outputs:
            return loss, outputs
        return loss

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    pred = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, pred),
        "macro_f1": f1_score(
            labels,
            pred,
            labels=list(range(len(LABELS))),
            average="macro",
            zero_division=0,
        ),
    }

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),

    # EXACTLY PRESERVED FROM REPRODUCED MD4
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=280,

    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,

    # Select the best checkpoint using the PRIMARY research metric.
    # Best checkpoint is selected manually after evaluating all epochs.
    # This avoids Trainer auto-reload key-name warnings seen in MD6.
    load_best_model_at_end=False,

    fp16=True,
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
)

print("Trainer GPU count:", trainer.args.n_gpu)
assert trainer.args.n_gpu == 2


model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainer GPU count: 2


In [4]:
# ============================================================
# TRAIN
# ============================================================

print("\n" + "=" * 78)
print("STARTING MD7 TRAINING")
print("=" * 78)

train_result = trainer.train()

# Preserve full Trainer history.
history_path = RESULT_DIR / "md7_log_history.json"
with history_path.open("w", encoding="utf-8") as f:
    json.dump(trainer.state.log_history, f, indent=2)

print("\n" + "=" * 78)
print("EPOCH VALIDATION HISTORY")
print("=" * 78)

for row in trainer.state.log_history:
    if "eval_macro_f1" in row:
        print(
            f"Epoch {row.get('epoch')}: "
            f"accuracy={row.get('eval_accuracy'):.6f} "
            f"macro_f1={row.get('eval_macro_f1'):.6f} "
            f"val_loss={row.get('eval_loss'):.6f}"
        )

print("\nAll 3 epoch checkpoints saved.")
print("Best checkpoint will be selected manually by validation Macro-F1.")



STARTING MD7 TRAINING


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.231499,1.214256,0.436757,0.434074
2,1.155553,1.198949,0.477838,0.473335
3,1.096134,1.210208,0.485946,0.475107


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


EPOCH VALIDATION HISTORY
Epoch 1.0: accuracy=0.436757 macro_f1=0.434074 val_loss=1.214256
Epoch 2.0: accuracy=0.477838 macro_f1=0.473335 val_loss=1.198949
Epoch 3.0: accuracy=0.485946 macro_f1=0.475107 val_loss=1.210208

All 3 epoch checkpoints saved.
Best checkpoint will be selected manually by validation Macro-F1.


In [5]:
# =========================================================
# EVALUATE ALL SAVED EPOCH CHECKPOINTS + SAVE AUDIT ARTIFACTS
# =========================================================

def softmax_np(logits):
    z = logits - logits.max(axis=1, keepdims=True)
    ez = np.exp(z)
    return ez / ez.sum(axis=1, keepdims=True)

def subset_metrics(true_ids, pred_ids, mask):
    y_true = true_ids[mask]
    y_pred = pred_ids[mask]
    return {
        "n": int(mask.sum()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(
            f1_score(
                y_true,
                y_pred,
                labels=list(range(len(LABELS))),
                average="macro",
                zero_division=0,
            )
        ),
    }

checkpoint_dirs = sorted(
    OUTPUT_DIR.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1]),
)

assert len(checkpoint_dirs) == 3, (
    f"Expected 3 checkpoints, found {len(checkpoint_dirs)}"
)

all_results = []

for epoch_idx, ckpt in enumerate(checkpoint_dirs, start=1):
    print("\n" + "-" * 78)
    print(f"EPOCH {epoch_idx}: {ckpt}")
    print("-" * 78)

    ckpt_model = AutoModelForSequenceClassification.from_pretrained(ckpt)

    eval_args = TrainingArguments(
        output_dir=str(RESULT_DIR / f"eval_epoch{epoch_idx}"),
        per_device_eval_batch_size=16,
        fp16=True,
        seed=SEED,
        data_seed=SEED,
        report_to="none",
    )

    eval_trainer = WeightedTrainer(
        model=ckpt_model,
        args=eval_args,
        eval_dataset=val_tok,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )

    pred_output = eval_trainer.predict(val_tok)
    logits = pred_output.predictions
    true_ids = pred_output.label_ids
    pred_ids = np.argmax(logits, axis=1)
    probs = softmax_np(logits)

    overall_accuracy = float(accuracy_score(true_ids, pred_ids))
    overall_macro_f1 = float(
        f1_score(
            true_ids,
            pred_ids,
            labels=list(range(len(LABELS))),
            average="macro",
            zero_division=0,
        )
    )

    per_class_f1 = f1_score(
        true_ids,
        pred_ids,
        labels=list(range(len(LABELS))),
        average=None,
        zero_division=0,
    )

    report = classification_report(
        true_ids,
        pred_ids,
        labels=list(range(len(LABELS))),
        target_names=LABELS,
        digits=6,
        zero_division=0,
        output_dict=True,
    )

    cm = confusion_matrix(
        true_ids,
        pred_ids,
        labels=list(range(len(LABELS))),
    ).tolist()

    short_result = subset_metrics(true_ids, pred_ids, short_mask)
    long_result = subset_metrics(true_ids, pred_ids, long_mask)

    result = {
        "epoch": epoch_idx,
        "checkpoint": str(ckpt),
        "accuracy": overall_accuracy,
        "macro_f1": overall_macro_f1,
        "per_class_f1": {
            label: float(score)
            for label, score in zip(LABELS, per_class_f1)
        },
        "classification_report": report,
        "confusion_matrix": cm,
        "le512": short_result,
        "gt512": long_result,
    }
    all_results.append(result)

    print("Accuracy:", overall_accuracy)
    print("Macro-F1:", overall_macro_f1)
    print("<=512 Macro-F1:", short_result["macro_f1"])
    print(">512 Macro-F1:", long_result["macro_f1"])
    print("Per-class F1:", result["per_class_f1"])

    pred_path = RESULT_DIR / f"md7_epoch{epoch_idx}_predictions.jsonl"
    with pred_path.open("w", encoding="utf-8") as f:
        for i, (true_id, pred_id, probability_row) in enumerate(
            zip(true_ids, pred_ids, probs)
        ):
            record = {
                "example_id": val_rows[i]["example_id"],
                "qid": val_rows[i]["qid"],
                "true_move": ID2LABEL[int(true_id)],
                "predicted_move": ID2LABEL[int(pred_id)],
                "original_distil_token_length": int(val_original_lengths[i]),
                "probabilities": {
                    label: float(probability_row[j])
                    for j, label in enumerate(LABELS)
                },
            }
            f.write(json.dumps(record) + "\n")

    del eval_trainer, ckpt_model
    gc.collect()
    torch.cuda.empty_cache()

best_result = max(all_results, key=lambda x: x["macro_f1"])

summary = {
    "experiment": "MD7_targeted_telling_weight",
    "research_question": (
        "Can increasing only the telling-class weight halfway from "
        "MD6 toward MD4 recover telling F1 without losing MD6 gains?"
    ),
    "model": MODEL_NAME,
    "max_length": MAX_LENGTH,
    "base_weight_alpha": BASE_WEIGHT_ALPHA,
    "telling_interpolation": TELLING_INTERPOLATION,
    "raw_class_weights": {
        label: float(w)
        for label, w in zip(LABELS, raw_class_weights)
    },
    "md6_class_weights": {
        label: float(w)
        for label, w in zip(LABELS, md6_class_weights)
    },
    "tuned_class_weights": {
        label: float(w)
        for label, w in zip(LABELS, class_weights)
    },
    "training_configuration": {
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 8,
        "effective_global_batch": 16,
        "num_train_epochs": 3,
        "weight_decay": 0.01,
        "warmup_steps": 280,
        "fp16": True,
        "seed": SEED,
        "test_used": False,
    },
    "baseline_md6_epoch2": {
        "accuracy": 0.5016216216216216,
        "macro_f1": 0.487684572655523,
    },
    "epochs": all_results,
    "best_epoch": best_result["epoch"],
    "best_accuracy": best_result["accuracy"],
    "best_macro_f1": best_result["macro_f1"],
}

summary_path = RESULT_DIR / "md7_summary.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

# Save a clean deployable copy of the winning checkpoint.
best_checkpoint = Path(best_result["checkpoint"])
best_model = AutoModelForSequenceClassification.from_pretrained(best_checkpoint)

best_model_dir = RESULT_DIR / "md7_best_model"
best_model.save_pretrained(best_model_dir)
tokenizer.save_pretrained(best_model_dir)

archive_path = shutil.make_archive(
    str(RESULT_DIR / "md7_best_model"),
    "gztar",
    root_dir=best_model_dir,
)

print("\n" + "=" * 78)
print("MD7 COMPLETE")
print("=" * 78)
print("Best epoch:", best_result["epoch"])
print("Best accuracy:", best_result["accuracy"])
print("Best Macro-F1:", best_result["macro_f1"])
print("Summary:", summary_path)
print("Best model archive:", archive_path)

print("\nDecision reference:")
print("MD6 Epoch 2 accuracy = 0.5016216216216216")
print("MD6 Epoch 2 Macro-F1 = 0.487684572655523")



------------------------------------------------------------------------------
EPOCH 1: /kaggle/working/md7_targeted_telling_weight/checkpoint-932
------------------------------------------------------------------------------


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Accuracy: 0.43675675675675674
Macro-F1: 0.4340735542435839
<=512 Macro-F1: 0.4528946079528622
>512 Macro-F1: 0.14003934562021123
Per-class F1: {'generic': 0.5771812080536913, 'probing': 0.3058103975535168, 'focus': 0.46319272125723737, 'telling': 0.3901098901098901}

------------------------------------------------------------------------------
EPOCH 2: /kaggle/working/md7_targeted_telling_weight/checkpoint-1864
------------------------------------------------------------------------------


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Accuracy: 0.47783783783783784
Macro-F1: 0.47333510724828226
<=512 Macro-F1: 0.4832287397480519
>512 Macro-F1: 0.1846539709239457
Per-class F1: {'generic': 0.5888456549935149, 'probing': 0.41012658227848103, 'focus': 0.5286274509803922, 'telling': 0.36574074074074076}

------------------------------------------------------------------------------
EPOCH 3: /kaggle/working/md7_targeted_telling_weight/checkpoint-2796
------------------------------------------------------------------------------


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Accuracy: 0.4859459459459459
Macro-F1: 0.4751072388118803
<=512 Macro-F1: 0.481223392061691
>512 Macro-F1: 0.2177594501718213
Per-class F1: {'generic': 0.5901639344262295, 'probing': 0.4043419267299864, 'focus': 0.5484351713859911, 'telling': 0.357487922705314}


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


MD7 COMPLETE
Best epoch: 3
Best accuracy: 0.4859459459459459
Best Macro-F1: 0.4751072388118803
Summary: /kaggle/working/md7_results/md7_summary.json
Best model archive: /kaggle/working/md7_results/md7_best_model.tar.gz

Decision reference:
MD6 Epoch 2 accuracy = 0.5016216216216216
MD6 Epoch 2 Macro-F1 = 0.487684572655523
